In [24]:
# Use this script after updating the deaths by county all time data in a local gdb but before uploading it to AGOL.
# You will use this script to take the deaths by county all time data and pivot it by incident type. Once you're done with
# this script (and any others that need to be run for this data), you can then upload both the tall and the pivoted
# data at once.

# Background:
# The deaths by county 5yr data is formatted such that each different combination of geography, year, age group, sex, and
# incident type are different records. This is fine for a table (though it makes for a very long table), but it's way too
# much data to display as a joined hosted feature layer. This data, the all time data, doesn't have too many records to
# work with as a joined hosted feature layer, but I wanted the layers to be structured and function similarly, so for those
# reasons it's pivoted for the map.

# Ideas for improvement:
# Bypass Excel. I'm only using it because this works well enough and I have other stuff to do.

In [25]:
# Used for re-shaping the table
import pandas as pd 

# Used for working with Excel
import openpyxl 

In [26]:
# You will need to have an ArcGIS Pro project open
aprx = arcpy.mp.ArcGISProject("CURRENT")

# In that project, you will need to have a map called "Map" (or change this name)
m = aprx.listMaps("Map")[0]

# In that map, you will need to have a table called "Work_Related_Deaths_by_County_All_Time" (or change this name)
deathsData = m.listTables("Work_Related_Deaths_by_County_All_Time")[0]

In [27]:
# Geodatabase location & where you want to save any Excel files
location = r"C:\Users\JohnsonN35\Local_work\Copied_from_local_work\MiTracking_Exploration"

# File name
file = "Deaths_all_time_pivoted_by_incident_number"

# Excel extension
ext = ".xlsx"

# Geodatabase name (this script assumes your gdb is in the same place as where you want to save any Excel files)
gdb = "MiTracking_Work_Related_Deaths_by_County_All_Time.gdb"

In [28]:
# Create a definition query for only the all-ages, etc. records

deathsDataDQ = '"Age_Group" = \'0\' And "Sex" = \'All\''

deathsData.definitionQuery = deathsDataDQ

In [29]:
# Create a pandas dataframe

fields = ['FIPS_Table', 'Time_Period', 'Incident_Type', 'Number_Deaths'] # Leaving out age group and sex

df = pd.DataFrame.from_records(data = arcpy.da.SearchCursor(deathsData, fields), columns = fields)

df.head()

,FIPS_Table,Time_Period,Incident_Type,Number_Deaths
0,26,2002-2021,0,2785
1,26,2002-2021,1,107
2,26,2002-2021,2,421
3,26,2002-2021,4,314
4,26,2002-2021,8,11


In [30]:
# Create a wide dataframe

df_wide = df.pivot(index = ['FIPS_Table', 'Time_Period'], columns = ['Incident_Type'], values = ['Number_Deaths'])

df_wide.reset_index(inplace = True)

df_wide.head()

FIPS_Table Time_Period Number_Deaths            ...                  
Incident_Type                                    0    1   11  ... 28 29    4   8  9
0                     26   2002-2021          2785  107  509  ...  1  2  314  11  7
1                  26000   2002-2021            11    0    2  ...  0  0    0   0  2
2                  26001   2002-2021             3    0    0  ...  0  0    1   0  0
3                  26003   2002-2021             5    0    1  ...  0  0    2   0  0
4                  26005   2002-2021            50    0   21  ...  0  0   12   0  0

[5 rows x 22 columns]

In [31]:
# Export to an Excel spreadsheet

name = file + "_pandas"

df_wide.to_excel(location + "/" + name + ext, sheet_name = 'Sheet1')

In [32]:
# Clean up the Excel file (we can't use the current table structure in a feature class)

from openpyxl import load_workbook

outpath = location + "/" + name + ext
wb = load_workbook(outpath)
ws = wb.active

# Start all the incident type fields with "Deaths"
# (i.e., if estimates go from cell D2 (4th column) - W2 (23rd column), put 23 next to the break)
for index, row in enumerate(ws.rows, start = 4):
        if index > 23: break
        ws.cell(row = 2, column = index).value = 'Deaths_' + ws.cell(row = 2, column = index).value

# Write the values of cells B1 (2nd column) - C1 (3rd column) into cells B2 - C2
for index, row in enumerate(ws.rows, start = 2):
        if index > 3: break
        ws.cell(row = 2, column = index).value = ws.cell(row = 1, column = index).value

# Unmerge estimate cells (causes issues with deleting rows in next step)
ws.unmerge_cells('D1:W1')

# Delete rows and columns we don't need and that can't be imported into a feature class
ws.delete_cols(1)
ws.delete_rows(1)
ws.delete_rows(2) # Once we delete the first row, the original third row becomes the second row
wb.save(outpath)

In [33]:
# Create a new table to hold the wide sheet

tableName = "Work_Related_Deaths_by_County_All_Time_pivoted"

arcpy.management.CreateTable(location + "/" + gdb, tableName)

<Result 'C:\\Users\\JohnsonN35\\Local_work\\Copied_from_local_work\\MiTracking_Exploration\\MiTracking_Work_Related_Deaths_by_County_All_Time.gdb\\Work_Related_Deaths_by_County_All_Time_pivoted'>

In [34]:
# Create the fields for the table; 
arcpy.management.AddFields(tableName, "\
FIPS_Table TEXT 'FIPS' 7 # #;\
Geographic_Area TEXT 'Geographic Area' 50 # #;\
Time_Period TEXT 'Time Period' 9 # # #;\
Deaths_0 SHORT 'Number of Deaths: All Incident Types' # # #;\
Deaths_1 SHORT 'Number of Deaths: Electrocution' # # #;\
Deaths_11 SHORT 'Number of Deaths: Motor Vehicle/Transportation-Related' # # #;\
Deaths_12 SHORT 'Number of Deaths: Homicide/Assault' # # #;\
Deaths_13 SHORT 'Number of Deaths: Drowning' # # #;\
Deaths_14 SHORT 'Number of Deaths: Suicide' # # #;\
Deaths_15 SHORT 'Number of Deaths: Toxic Exposure' # # #;\
Deaths_16 SHORT 'Number of Deaths: Struck by Object' # # #;\
Deaths_18 SHORT 'Number of Deaths: Fire/Explosion' # # #;\
Deaths_2 SHORT 'Number of Deaths: Fall-Related' # # #;\
Deaths_20 SHORT 'Number of Deaths: Heat/Cold-Related' # # #;\
Deaths_24 SHORT 'Number of Deaths: Animal' # # #;\
Deaths_25 SHORT 'Number of Deaths: Aircraft' # # #;\
Deaths_26 SHORT 'Number of Deaths: Asphyxiation' # # #;\
Deaths_27 SHORT 'Number of Deaths: Drug Overdose' # # #;\
Deaths_28 SHORT 'Number of Deaths: Infectious Disease' # # #;\
Deaths_29 SHORT 'Number of Deaths: Medical' # # #;\
Deaths_4 SHORT 'Number of Deaths: Machine-Related' # # #;\
Deaths_8 SHORT 'Number of Deaths: Other' # # #;\
Deaths_9 SHORT 'Number of Deaths: Unknown' # # #;\
")

<Result 'Work_Related_Deaths_by_County_All_Time_pivoted'>

In [35]:
# Append wide sheet to new table

spreadsheet = location + "\\" + name + ext + "\\Sheet1$"

fieldMapping = '\
    FIPS_Table "FIPS" true true false 7 Text 0 0,First,#,spreadsheet,FIPS_Table,0,254;\
    Geographic_Area "Geographic Area" true true false 50 Text 0 0,First,#,spreadsheet,Geographic_Area,0,254;\
    Time_Period "Time Period" true true false 9 Text 0 0,First,#,spreadsheet,Time_Period,0,254;\
    Deaths_0 "Number of Deaths: All Incident Types" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_0,-1,-1;\
    Deaths_1 "Number of Deaths: Electrocution" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_1,-1,-1;\
    Deaths_11 "Number of Deaths: Motor Vehicle/Transportation-Related" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_11,-1,-1;\
    Deaths_12 "Number of Deaths: Homicide/Assault" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_12,-1,-1;\
    Deaths_13 "Number of Deaths: Drowning" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_13,-1,-1;\
    Deaths_14 "Number of Deaths: Suicide" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_14,-1,-1;\
    Deaths_15 "Number of Deaths: Toxic Exposure" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_15,-1,-1;\
    Deaths_16 "Number of Deaths: Struck by Object" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_16,-1,-1;\
    Deaths_18 "Number of Deaths: Fire/Explosion" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_18,-1,-1;\
    Deaths_2 "Number of Deaths: Fall-Related" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_2,-1,-1;\
    Deaths_20 "Number of Deaths: Heat/Cold-Related" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_20,-1,-1;\
    Deaths_24 "Number of Deaths: Animal" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_24,-1,-1;\
    Deaths_25 "Number of Deaths: Aircraft" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_25,-1,-1;\
    Deaths_26 "Number of Deaths: Asphyxiation" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_26,-1,-1;\
    Deaths_27 "Number of Deaths: Drug Overdose" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_27,-1,-1;\
    Deaths_28 "Number of Deaths: Infectious Disease" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_28,-1,-1;\
    Deaths_29 "Number of Deaths: Medical" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_29,-1,-1;\
    Deaths_4 "Number of Deaths: Machine-Related" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_4,-1,-1;\
    Deaths_8 "Number of Deaths: Other" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_8,-1,-1;\
    Deaths_9 "Number of Deaths: Unknown" true true false 2 Short 0 0,First,#,spreadsheet,Deaths_9,-1,-1\
    '

arcpy.management.Append(spreadsheet, tableName, "NO_TEST", fieldMapping)

<Result 'Work_Related_Deaths_by_County_All_Time_pivoted'>

In [ ]:
# Create local copy of the appropriate geography layer, join by FIPS, and field-calculate geographic area names into pivoted layer:
# https://mdhhs.maps.arcgis.com/home/item.html?id=ec983f14c3394a53a67711b9ffa219f2